# Hybrid experiment: draft + refine vs. full Wan2.2-Animate

Runs `experiment_hybrid.py` on Colab for one template role and one photo: full Wan2.2-Animate-14B
baseline, a draft, SDEdit-style refinement at several strengths, then timings, metrics and a
side-by-side video. See `CLAUDE.md` for the reasoning.

**GPU.** Runtime → Change runtime type:
- **H100 or A100 80 GB**: everything runs as written.
- **A100 40 GB**: the notebook turns on `--offload` automatically. Slower, untested; may run out of memory at 720p.
- **T4 / L4**: too small for the 14B baseline. Only the refine part can run, with `EXISTING_BASELINE` set to a baseline made elsewhere.

**Google Drive layout** (inputs; results are written back there):
```
MyDrive/video_versioner/
  pipeline.py
  experiment_hybrid.py
  templates/alors_on_danse.json   # paths inside are relative to video_versioner/
  templates/alors_on_danse.mp4
  photos/alice.jpg
```
`prepared/` and `experiments/` are created next to them, so later sessions skip the preprocessing.

In [ ]:
#@title Settings
DRIVE_DIR = "/content/drive/MyDrive/video_versioner"  #@param {type:"string"}
TEMPLATE = "alors_on_danse"  #@param {type:"string"}
ROLE = "keyboard"  #@param {type:"string"}
PHOTO = "photos/alice.jpg"  #@param {type:"string"}
REFINE_MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"  #@param ["Wan-AI/Wan2.1-T2V-1.3B-Diffusers", "Wan-AI/Wan2.1-T2V-14B-Diffusers"]
STRENGTHS = "0.3 0.5 0.7"  #@param {type:"string"}
STEPS = 40  #@param {type:"integer"}
#@markdown Leave empty to generate the baseline; or a path relative to `DRIVE_DIR` to reuse one.
EXISTING_BASELINE = ""  #@param {type:"string"}
#@markdown Identity scores need insightface (slow to install).
IDENTITY_SCORES = True  #@param {type:"boolean"}

## 1. Drive, workspace and GPU

In [ ]:
import shutil
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
WORK = Path("/content/video_versioner")  # local copy: much faster than working on Drive
WAN_REPO = Path("/content/Wan2.2")
CKPT = Path("/content/Wan2.2-Animate-14B")

for rel in ["pipeline.py", "experiment_hybrid.py", f"templates/{TEMPLATE}.json", PHOTO]:
    assert (DRIVE / rel).exists(), f"missing {DRIVE / rel}"
shutil.copytree(DRIVE, WORK, dirs_exist_ok=True)
%cd {WORK}

PREPARED = Path("prepared") / TEMPLATE
NEED_PREPARE = not (PREPARED / "manifest.json").exists()
NEED_WAN = NEED_PREPARE or not EXISTING_BASELINE

assert torch.cuda.is_available(), "no GPU: Runtime → Change runtime type"
gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 2**30
OFFLOAD = gpu_gb < 70
print(f"{gpu.name}, {gpu_gb:.0f} GB → offload: {OFFLOAD}")
print(f"preprocessing needed: {NEED_PREPARE}, Wan2.2-Animate needed: {NEED_WAN}")
if NEED_WAN and gpu_gb < 30:
    print("WARNING: this GPU is too small for Wan2.2-Animate-14B. Prepare the template and the "
          "baseline on a bigger GPU, then set EXISTING_BASELINE.")
!df -h /content | tail -1

## 2. Install

Wan2.2 needs `flash_attn`, which usually compiles from source on Colab: this can take a long time.
If pip replaces Colab's `torch`, restart the runtime (Runtime → Restart session) and run again
from the top; already-installed packages are kept.

In [ ]:
%%bash
set -e
if [ ! -d /content/Wan2.2 ]; then git clone --depth 1 https://github.com/Wan-Video/Wan2.2.git /content/Wan2.2; fi
cd /content/Wan2.2
grep -v flash_attn requirements.txt > /tmp/wan_requirements.txt
pip install -q -r /tmp/wan_requirements.txt
if [ -f requirements_animate.txt ]; then pip install -q -r requirements_animate.txt; fi
pip install -q "diffusers>=0.33" transformers accelerate ftfy imageio-ffmpeg opencv-python huggingface_hub

In [ ]:
if NEED_WAN:
    !MAX_JOBS=8 pip install flash-attn --no-build-isolation
if IDENTITY_SCORES:
    !pip install -q insightface onnxruntime-gpu

## 3. Wan2.2-Animate-14B weights (only if preprocessing or the baseline must run)

In [ ]:
from huggingface_hub import snapshot_download

if NEED_WAN:
    snapshot_download("Wan-AI/Wan2.2-Animate-14B", local_dir=CKPT)
    !du -sh {CKPT}
else:
    print("skipped: template already prepared and baseline given")

## 4. Prepare the template (once; saved to Drive)

In [ ]:
if NEED_PREPARE:
    !python pipeline.py prepare --template templates/{TEMPLATE}.json --wan-repo {WAN_REPO} --ckpt {CKPT} --out {PREPARED}
    shutil.copytree(PREPARED, DRIVE / PREPARED, dirs_exist_ok=True)
else:
    print(f"using {DRIVE / PREPARED}")

## 5. Run the experiment

In [ ]:
import shlex

model_tag = REFINE_MODEL.split("/")[-1].replace("-Diffusers", "")
OUT = Path("experiments") / f"{TEMPLATE}_{ROLE}_{Path(PHOTO).stem}_{model_tag}"
cmd = ["python", "experiment_hybrid.py", "--prepared", PREPARED, "--role", ROLE, "--photo", PHOTO,
       "--out", OUT, "--refine-model", REFINE_MODEL, "--steps", STEPS, "--strengths", *STRENGTHS.split()]
if EXISTING_BASELINE:
    cmd += ["--baseline", EXISTING_BASELINE]
else:
    cmd += ["--wan-repo", WAN_REPO, "--ckpt", CKPT]
if OFFLOAD:
    cmd.append("--offload")
CMD = shlex.join(map(str, cmd))
print(CMD)
!{CMD}
shutil.copytree(OUT, DRIVE / OUT, dirs_exist_ok=True)
print(f"results saved to {DRIVE / OUT}")

## 6. Results

In [ ]:
import json
from IPython.display import Markdown, Video, display

report = json.loads((OUT / "report.json").read_text())
fmt = lambda v, spec: "-" if v is None else format(v, spec)
rows = ["| variant | identity | face rate | flicker | seam flicker | PSNR vs baseline |", "|---|---|---|---|---|---|"]
rows += [f"| {n} | {fmt(m['identity'], '.3f')} | {fmt(m['face_rate'], '.0%')} | {m['flicker']:.2f} | "
         f"{fmt(m['seam_flicker'], '.2f')} | {fmt(m['psnr_vs_baseline'], '.1f')} |"
         for n, m in report["metrics"].items()]
rows += ["", "| step | seconds |", "|---|---|"] + [f"| {n} | {s:.0f} |" for n, s in report["seconds"].items()]
display(Markdown("\n".join(rows)))
display(Video(str(OUT / "side_by_side.mp4"), embed=True, width=1200))

**Reading the results**
- The default draft is the baseline degraded to look like a game render: it already has perfect
  motion and identity, so this is the optimistic case.
- The refiner doesn't see the photo: identity scores are a lower bound.
- The baseline time includes loading the 14B model; the refiner's load is reported separately.
- Watch the side-by-side video: the numbers don't capture realism.